# 02 - Orbit Prediction

Train LSTM and Transformer models to predict spacecraft orbits.  
Benchmark against Kepler (two-body) baseline.

In [ ]:
import sys
sys.path.insert(0, '..')

import os
os.chdir('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from src.data.ssc_client import SSCClient
from src.data.preprocessing import OrbitPreprocessor
from src.data.dataset import create_dataloaders
from src.models.lstm import OrbitLSTM, OrbitLSTMDirect
from src.models.transformer import OrbitTransformer
from src.models.baseline_sgp4 import evaluate_baseline
from src.training.train import Trainer
from src.training.evaluate import evaluate_pytorch_model, comparison_table
from src.utils.visualization import (
    plot_3d_orbit_plotly, plot_prediction_error, 
    plot_model_comparison, plot_training_history
)

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

## 1. Load and Preprocess Data

In [ ]:
# Fetch ISS data (start with 3 months for fast iteration)
ssc = SSCClient()
iss_raw = ssc.fetch_positions('iss', '2024-01-01', '2024-04-01')
print(f'Raw data: {iss_raw.shape}')
iss_raw.head()

In [ ]:
# Preprocess: derive velocity, normalize
preprocessor = OrbitPreprocessor()
iss_df = preprocessor.preprocess(iss_raw, 'iss')

print(f'Preprocessed: {iss_df.shape}')
print(f'Columns: {list(iss_df.columns)}')
print(f'\nNormalization stats:')
for k, v in preprocessor.stats['iss']['mean'].items():
    print(f'  {k}: mean={v:.2f}, std={preprocessor.stats["iss"]["std"][k]:.2f}')

In [ ]:
# Create sliding windows: 24h input -> 6h prediction
HORIZON_HOURS = 6
inputs, targets, timestamps = preprocessor.create_sliding_windows(
    iss_df, input_hours=24, horizon_hours=HORIZON_HOURS, stride_hours=2
)
print(f'Input shape:  {inputs.shape}  (samples, input_steps, features)')
print(f'Target shape: {targets.shape}  (samples, horizon_steps, xyz)')

# Split
splits = preprocessor.temporal_split(inputs, targets, timestamps)
for name, (x, y) in splits.items():
    print(f'{name}: {x.shape[0]} samples')

## 2. Kepler Baseline

In [ ]:
test_inputs, test_targets = splits['test']

# Extract last position and velocity from each input window
# These are normalized — denormalize for baseline
denorm = lambda x: preprocessor.denormalize(x, 'iss')

# Initial positions (last timestep of input)
test_pos = denorm(test_inputs[:, -1:, :3]).squeeze(1)
test_vel = test_inputs[:, -1, 3:6]  # Already in normalized velocity

# Denormalize velocity
vel_stats = preprocessor.stats['iss']
test_vel_km = np.zeros_like(test_vel)
for i, col in enumerate(['vx_gse', 'vy_gse', 'vz_gse']):
    if col in vel_stats['mean']:
        test_vel_km[:, i] = test_vel[:, i] * vel_stats['std'][col] + vel_stats['mean'][col]

# Denormalize targets
test_targets_km = denorm(test_targets)

# Evaluate
baseline_results = evaluate_baseline(test_pos, test_vel_km, test_targets_km, dt_seconds=60)
print('Kepler Baseline Results:')
for k, v in baseline_results.items():
    if v is not None:
        print(f'  {k}: {v:.2f}')

## 3. Train LSTM

In [ ]:
input_dim = inputs.shape[-1]
output_dim = targets.shape[-1]
horizon_steps = targets.shape[1]

# Start with the direct (non-autoregressive) model for speed
lstm_model = OrbitLSTMDirect(
    input_dim=input_dim,
    hidden_dim=128,
    num_layers=2,
    horizon=horizon_steps,
    output_dim=output_dim,
    dropout=0.1,
)

print(f'LSTM parameters: {sum(p.numel() for p in lstm_model.parameters()):,}')

loaders = create_dataloaders(splits, batch_size=64)
trainer = Trainer(lstm_model, device=device)
lstm_history = trainer.train(loaders['train'], loaders['val'], model_name='lstm_iss')

In [ ]:
plot_training_history(lstm_history, title='LSTM Training')

In [ ]:
lstm_results = evaluate_pytorch_model(lstm_model, loaders['test'], denormalize_fn=denorm, device=device)
print('LSTM Results:')
for k, v in lstm_results.items():
    if k != 'error_over_time':
        print(f'  {k}: {v:.2f}')

## 4. Train Transformer

In [ ]:
transformer_model = OrbitTransformer(
    input_dim=input_dim,
    d_model=128,
    nhead=4,
    num_encoder_layers=2,
    num_decoder_layers=2,
    dim_feedforward=256,
    horizon=horizon_steps,
    output_dim=output_dim,
    dropout=0.1,
)

print(f'Transformer parameters: {sum(p.numel() for p in transformer_model.parameters()):,}')

trainer = Trainer(transformer_model, device=device)
transformer_history = trainer.train(loaders['train'], loaders['val'], model_name='transformer_iss')

In [ ]:
plot_training_history(transformer_history, title='Transformer Training')

In [ ]:
transformer_results = evaluate_pytorch_model(
    transformer_model, loaders['test'], denormalize_fn=denorm, device=device
)
print('Transformer Results:')
for k, v in transformer_results.items():
    if k != 'error_over_time':
        print(f'  {k}: {v:.2f}')

## 5. Compare All Models

In [ ]:
all_results = {
    'Kepler Baseline': baseline_results,
    'LSTM': lstm_results,
    'Transformer': transformer_results,
}

print(comparison_table(all_results))

In [ ]:
# Error growth over prediction horizon
plot_model_comparison(all_results, title='ISS Orbit Prediction: Error vs Horizon')

In [ ]:
# Visualize a sample prediction
lstm_model.eval()
with torch.no_grad():
    sample_input = torch.from_numpy(test_inputs[:1]).to(device)
    sample_pred = lstm_model(sample_input).cpu().numpy()

pred_km = denorm(sample_pred[0])
actual_km = denorm(test_targets[:1])[0]

fig = plot_3d_orbit_plotly(actual_km, pred_km, title='LSTM: Predicted vs Actual (6h horizon)')
fig.show()

## 6. GPU Training Results (RunPod, dual RTX 5090)

Full-scale training on 3 years of 1-min resolution data (2023-2025), 24h input → 6h prediction.

| Model | ISS (LEO) | DSCOVR (L1) | MMS-1 (HEO) |
|-------|-----------|-------------|-------------|
| **LSTM** | **125.80 km** | **12,797 km** | **18,683 km** |
| Transformer | 294.69 km | 13,517 km | 19,237 km |

**Key findings:**
- LSTM consistently outperforms Transformer across all spacecraft
- ISS (LEO) is most predictable — tight orbit, dense data (1.58M points)
- DSCOVR/MMS-1 have much larger errors due to complex orbital dynamics at L1 and in the magnetosphere
- Training time: ~40 min total for all 6 model/spacecraft combinations

See `scripts/train_gpu.py` for the full training pipeline and `datamatters24/orbital-chaos-predictor` on HF for checkpoints.